# 01. Limpieza de datos y metodología CRISP-DM
## Proyecto final — Modelo predictivo Mundial 2026

**Responsable de esta sección (Persona 2):** limpieza de datos, selección de variables, y redacción de
*Business Understanding*, *Data Understanding* y *Data Preparation*.

Este notebook toma los archivos crudos descargados por Persona 1 (documentados en
`documentacion_fuentes_mundial_2026.md`) y produce el dataset limpio que usará Persona 3 para entrenar
el modelo predictivo (`02_modelo_predictivo.ipynb`).

**Salidas de este notebook:**

- `data/processed/clean_model_dataset.csv` — dataset a nivel de partido, con variables de forma
  reciente, fuerza relativa histórica (Elo) y contexto del partido, listo para modelar.
- `data/processed/team_features_2026.csv` — foto actual (ranking FIFA, Elo, forma reciente) de cada
  selección, usada como punto de partida para simular el Mundial 2026 hacia adelante.


## 1. Business Understanding

**Objetivo del negocio/proyecto:** construir, de forma reproducible y justificable con datos, un
modelo que estime la probabilidad de que cada selección participante en el Mundial 2026 llegue a
semifinales, final, y finalmente prediga el podio completo: campeón, subcampeón, 3° y 4° lugar.

**Por qué importa:** la evaluación premia el rigor metodológico (trazabilidad de los datos, decisiones
de limpieza justificadas, y un modelo explicable) por encima de acertar el resultado final. El Mundial
2026 se encuentra en fase eliminatoria y la final es el 19 de julio, por lo que el modelo debe poder
generar una predicción de podio *antes* de esa fecha, usando información disponible hasta el momento
del corte de datos (8 de julio de 2026).

**Criterios de éxito:**

1. El dataset limpio debe permitir entrenar un modelo de resultado de partido (o de fuerza relativa
   entre selecciones) sin fugas de información (*data leakage*) hacia el futuro.
2. Cada variable incluida debe estar justificada: de dónde sale, qué mide, y por qué es relevante para
   predecir desempeño en un torneo eliminatorio.
3. El dataset debe ser reutilizable tanto para entrenar (histórico) como para simular el Mundial 2026
   hacia adelante (foto actual de cada selección).

**Restricciones conocidas del negocio/proyecto:**

- Fecha de entrega: sábado 11 de julio, 09:59 horas — el pipeline debe ser reproducible en minutos, no
  en horas.
- El ranking FIFA es una **foto del 8 de julio de 2026**, no una serie histórica por fecha de partido;
  esto se documenta como limitación en Data Preparation. El rating Elo, en cambio, sí resultó ser una
  serie histórica completa (ver sección 3.5), lo cual mejora la calidad del modelo.


## 2. Data Understanding

Fuentes utilizadas (ver detalle completo en `documentacion_fuentes_mundial_2026.md`):

| Archivo | Descripción | Nivel |
|---|---|---|
| `results.csv` | Historial de partidos internacionales (Kaggle, M. Jürisoo) | partido |
| `shootouts.csv` | Resultados de tandas de penales | partido |
| `fifa_ranking_men_2026_07_08.csv` | Ranking oficial FIFA, foto única al 2026-07-08 | selección |
| `elo_ratings_2026_07_08.csv` | Rating Elo **histórico**: una fila por selección cada vez que su rating cambió. Formato de fecha mixto (`YYYY-MM-DD` / `MM/DD/YYYY`); el último rating real es de diciembre de 2025, no de julio 2026 | selección + fecha |

Los archivos `goalscorers.csv` y los de Transfermarkt (`opcional/`) quedan documentados pero **no se
usan** en la primera versión del modelo, siguiendo la recomendación de Persona 1 de mantener el dataset
base simple y justificable.

A continuación se cargan los archivos crudos y se hace una inspección inicial: dimensiones, tipos de
dato, valores nulos, cobertura de fechas y consistencia de nombres de selecciones entre fuentes.


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 50)

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

results = pd.read_csv(RAW_DIR / "results.csv", encoding="utf-8", parse_dates=["date"])
shootouts = pd.read_csv(RAW_DIR / "shootouts.csv", encoding="utf-8", parse_dates=["date"])
fifa_rank = pd.read_csv(RAW_DIR / "fifa_ranking_men_2026_07_08.csv", encoding="utf-8")
elo = pd.read_csv(RAW_DIR / "elo_ratings_2026_07_08.csv", encoding="utf-8")

# elo_ratings.csv mezcla dos formatos de fecha (YYYY-MM-DD para partidos antiguos, MM/DD/YYYY para
# los más recientes). read_csv con parse_dates falla silenciosamente ante formato mixto y deja la
# columna como texto, así que se parsea aparte con format="mixed".
elo["date"] = pd.to_datetime(elo["date"], format="mixed", dayfirst=False)

print("results:", results.shape)
print("shootouts:", shootouts.shape)
print("fifa_rank:", fifa_rank.shape)
print("elo:", elo.shape)


results: (49505, 9)
shootouts: (682, 5)
fifa_rank: (211, 13)
elo: (6678, 4)


In [2]:
# Inspección de tipos y nulos por dataset
for name, df in [("results", results), ("shootouts", shootouts),
                  ("fifa_rank", fifa_rank), ("elo", elo)]:
    print(f"\n--- {name} ---")
    print(df.dtypes)
    print(df.isna().sum()[df.isna().sum() > 0])



--- results ---
date          datetime64[us]
home_team                str
away_team                str
home_score           float64
away_score           float64
tournament               str
city                     str
country                  str
neutral                 bool
dtype: object
home_score    4
away_score    4
dtype: int64

--- shootouts ---
date             datetime64[us]
home_team                   str
away_team                   str
winner                      str
first_shooter               str
dtype: object
first_shooter    422
dtype: int64

--- fifa_rank ---
rank                  int64
prev_rank             int64
ranking_movement      int64
team                    str
country_code            str
confederation           str
total_points        float64
prev_points         float64
rated_matches         int64
id_team               int64
id_confederation      int64
gender                int64
ranking_status        int64
dtype: object
Series([], dtype: int64)

--- elo ---
d

In [3]:
# Cobertura temporal
print("results.csv:", results["date"].min(), "a", results["date"].max())
print("elo_ratings.csv:", elo["date"].min(), "a", elo["date"].max())
print("\nPartidos por torneo (top 15):")
print(results["tournament"].value_counts().head(15))
print("\n% de partidos en sede neutral:", round(results["neutral"].mean() * 100, 1), "%")


results.csv: 1872-11-30 00:00:00 a 2026-07-11 00:00:00
elo_ratings.csv: 1872-11-30 00:00:00 a 2025-12-13 00:00:00

Partidos por torneo (top 15):
tournament
Friendly                                18388
FIFA World Cup qualification             8771
UEFA Euro qualification                  2824
African Cup of Nations qualification     2327
FIFA World Cup                           1064
Copa América                              869
African Cup of Nations                    845
AFC Asian Cup qualification               829
UEFA Nations League                       658
CECAFA Cup                                620
CFU Caribbean Cup qualification           606
Merdeka Tournament                        599
British Home Championship                 523
CONCACAF Nations League                   422
AFC Asian Cup                             421
Name: count, dtype: int64

% de partidos en sede neutral: 26.6 %


In [4]:
# Consistencia de nombres de selección entre fuentes.
# elo_ratings.csv trae espacios "non-breaking" (\xa0) en varios nombres (artefacto de scraping web
# de eloratings.net), por lo que se limpian antes de comparar; si no, cualquier merge por nombre
# pierde filas silenciosamente aunque el nombre "se vea" igual impreso en pantalla.
def strip_nbsp(name):
    return name.replace("\xa0", " ").strip() if isinstance(name, str) else name

elo["team"] = elo["team"].apply(strip_nbsp)

teams_results = set(results["home_team"]).union(results["away_team"])
teams_fifa = set(fifa_rank["team"])
teams_elo = set(elo["team"])

print("Selecciones en fifa_rank (211 miembros FIFA) sin match directo en results.csv:")
print(sorted(teams_fifa - teams_results))

print("\nSelecciones en fifa_rank sin match directo en elo_ratings.csv:")
print(sorted(teams_fifa - teams_elo))


Selecciones en fifa_rank (211 miembros FIFA) sin match directo en results.csv:
['Cabo Verde', 'China PR', 'Chinese Taipei', 'Congo DR', 'Czechia', "Côte d'Ivoire", 'DPR Korea', 'IR Iran', 'Korea Republic', 'Kyrgyz Republic', 'St. Kitts and Nevis', 'St. Lucia', 'St. Vincent / Grenadines', 'The Gambia', 'Türkiye', 'US Virgin Islands', 'USA']

Selecciones en fifa_rank sin match directo en elo_ratings.csv:
['American Samoa', 'Cabo Verde', 'China PR', 'Chinese Taipei', 'Congo DR', "Côte d'Ivoire", 'DPR Korea', 'IR Iran', 'Korea Republic', 'Kyrgyz Republic', 'Macau', 'Republic of Ireland', 'St. Kitts and Nevis', 'St. Lucia', 'St. Vincent / Grenadines', 'São Tomé and Príncipe', 'The Gambia', 'Timor-Leste', 'Türkiye', 'USA']


## 3. Data Preparation

Pasos aplicados, en orden, y la razón de cada uno:

1. **Normalización de nombres de selección.** Las tres fuentes usan convenciones de nombre distintas
   para el mismo equipo (además de espacios non-breaking en `elo_ratings.csv`). Sin esto, cualquier
   `merge` pierde filas silenciosamente.
2. **Integración de tandas de penales.** `shootouts.csv` define quién ganó un partido que terminó
   empatado en tiempo reglamentario (relevante en fases eliminatorias).
3. **Definición del target.** Resultado del partido desde la perspectiva del equipo local
   (`home_win` / `draw` / `away_win`), y diferencia de goles.
4. **Ingeniería de variables de forma reciente**, calculadas de forma *cronológica* (solo con
   información anterior a la fecha del partido) para evitar fuga de información hacia el futuro.
5. **Variables de contexto del partido**: sede neutral, tipo/importancia del torneo.
6. **Fuerza relativa Elo pre-partido**, unida por fecha con un *as-of merge* (solo ratings anteriores
   al partido, nunca posteriores). El ranking FIFA, al ser una única foto, solo se usa para la foto
   actual de cada selección, no como variable de entrenamiento histórica.
7. **Selección final de variables** y exportación de los dos archivos de salida.


### 3.1 Normalización de nombres de selección

El diccionario de alias se construyó **verificando contra los archivos reales** (no supuestos):
para cada selección de las 211 del ranking FIFA se comparó su nombre normalizado (sin tildes, sin
espacios non-breaking, minúsculas) contra `results.csv` y `elo_ratings.csv`, y se resolvieron a mano
las discrepancias reales que quedaron. La mayoría son variantes del nombre oficial FIFA
(`"USA"`, `"IR Iran"`, `"Korea Republic"`, `"Türkiye"`, `"Cabo Verde"`...) frente al nombre más común
usado en `results.csv`/`elo_ratings.csv` (`"United States"`, `"Iran"`, `"South Korea"`, `"Turkey"`,
`"Cape Verde"`...).


In [5]:
# Nombre de la fuente -> nombre estándar del proyecto (verificado contra los 3 archivos reales).
# El nombre estándar elegido es, en cada caso, el que ya comparten results.csv y elo_ratings.csv.
TEAM_NAME_MAP = {
    # Variantes del nombre oficial FIFA
    "USA": "United States",
    "IR Iran": "Iran",
    "Korea Republic": "South Korea",
    "DPR Korea": "North Korea",
    "Côte d'Ivoire": "Ivory Coast",
    "Cabo Verde": "Cape Verde",
    "Congo DR": "DR Congo",
    "Czechia": "Czech Republic",
    "China PR": "China",
    "Chinese Taipei": "Taiwan",
    "Kyrgyz Republic": "Kyrgyzstan",
    "The Gambia": "Gambia",
    "Türkiye": "Turkey",
    "US Virgin Islands": "United States Virgin Islands",
    "St. Kitts and Nevis": "Saint Kitts and Nevis",
    "St. Lucia": "Saint Lucia",
    "St. Vincent / Grenadines": "Saint Vincent and the Grenadines",
    # Variantes específicas de elo_ratings.csv
    "Democratic Republic of Congo": "DR Congo",
    "Republic of Ireland": "Ireland",
}

def normalize_team(name: str) -> str:
    if not isinstance(name, str):
        return name
    name = strip_nbsp(name)
    return TEAM_NAME_MAP.get(name, name)

for df, cols in [(results, ["home_team", "away_team"]),
                  (shootouts, ["home_team", "away_team"]),
                  (fifa_rank, ["team"]),
                  (elo, ["team"])]:
    for c in cols:
        df[c] = df[c].apply(normalize_team)

# Re-chequeo de cobertura tras normalizar: selecciones FIFA que de verdad quedan sin match.
teams_results = set(results["home_team"]).union(results["away_team"])
teams_fifa = set(fifa_rank["team"])
teams_elo = set(elo["team"])

print("FIFA sin match en results tras normalizar:", sorted(teams_fifa - teams_results))
print("FIFA sin match en elo tras normalizar:", sorted(teams_fifa - teams_elo))


FIFA sin match en results tras normalizar: []
FIFA sin match en elo tras normalizar: ['American Samoa', 'Macau', 'São Tomé and Príncipe', 'Timor-Leste']


Las selecciones que quedan sin match (p. ej. Samoa Americana, Macao, Timor Oriental en `elo_ratings.csv`)
son equipos de rating muy bajo, sin opción real de clasificar al Mundial 2026; se documentan como
limitación menor y quedarán con Elo nulo en vez de forzar un alias sin verificar.


### 3.2 Integración de tandas de penales y definición del target


In [6]:
shootouts_slim = shootouts[["date", "home_team", "away_team", "winner"]].rename(
    columns={"winner": "shootout_winner"}
)

matches = results.merge(
    shootouts_slim, on=["date", "home_team", "away_team"], how="left"
)

matches["decided_by_penalties"] = matches["shootout_winner"].notna()

def match_result(row):
    if row["home_score"] > row["away_score"]:
        return "home_win"
    if row["home_score"] < row["away_score"]:
        return "away_win"
    # Empate en tiempo reglamentario: si hubo penales, el resultado real lo define la tanda.
    if row["decided_by_penalties"]:
        return "home_win" if row["shootout_winner"] == row["home_team"] else "away_win"
    return "draw"

matches["result"] = matches.apply(match_result, axis=1)
matches["goal_diff"] = matches["home_score"] - matches["away_score"]

matches["result"].value_counts(normalize=True).round(3)


result
home_win    0.497
away_win    0.289
draw        0.214
Name: proportion, dtype: float64

### 3.3 Variables de contexto del partido

`neutral` ya viene en el dataset original. Se agrega una variable ordinal simple de importancia del
torneo, porque no es lo mismo un amistoso que un partido de Mundial o de clasificatoria: el nivel de
esfuerzo y de información que aporta el resultado es distinto.


In [7]:
def tournament_tier(t: str) -> int:
    t = (t or "").lower()
    if "world cup" in t and "qualif" not in t:
        return 4
    if "cup" in t or "championship" in t or "copa" in t or "euro" in t:
        return 3
    if "qualif" in t:
        return 2
    return 1  # amistosos y torneos menores

matches["tournament_tier"] = matches["tournament"].apply(tournament_tier)
matches["neutral"] = matches["neutral"].astype(int)
matches[["tournament", "tournament_tier"]].drop_duplicates().sort_values("tournament_tier", ascending=False).head(10)


,tournament,tournament_tier
1490,FIFA World Cup,4
30497,Viva World Cup,4
196,Copa Newton,3
29,British Home Championship,3
385,Far Eastern Championship Games,3
450,Copa Roca,3
320,Copa Premio Honor Uruguayo,3
237,Copa Premio Honor Argentino,3
10434,Copa Félix Bogado,3
10925,Real Madrid 75th Anniversary Cup,3


### 3.4 Forma reciente por selección (sin fuga de información)

Se transforma `matches` a formato largo (una fila por selección y partido) para poder calcular, por
selección y en orden cronológico, estadísticas de las últimas *N* apariciones **usando `shift` antes
de agregar**, de modo que la fila del partido *t* solo vea información de partidos anteriores a *t*.


In [8]:
matches = matches.sort_values("date").reset_index(drop=True)
matches["match_id"] = matches.index

home_long = matches[["match_id", "date", "home_team", "home_score", "away_score", "result"]].copy()
home_long.columns = ["match_id", "date", "team", "gf", "ga", "result"]
home_long["is_home"] = 1
home_long["points"] = home_long["result"].map({"home_win": 3, "draw": 1, "away_win": 0})

away_long = matches[["match_id", "date", "away_team", "away_score", "home_score", "result"]].copy()
away_long.columns = ["match_id", "date", "team", "gf", "ga", "result"]
away_long["is_home"] = 0
away_long["points"] = away_long["result"].map({"away_win": 3, "draw": 1, "home_win": 0})

team_long = pd.concat([home_long, away_long], ignore_index=True)
team_long = team_long.sort_values(["team", "date"]).reset_index(drop=True)

WINDOW = 10
grouped = team_long.groupby("team", group_keys=False)

# shift(1) antes de rolling: la fila del partido actual nunca incluye su propio resultado.
team_long["form_points_avg"] = grouped["points"].apply(lambda s: s.shift(1).rolling(WINDOW, min_periods=1).mean())
team_long["form_gf_avg"] = grouped["gf"].apply(lambda s: s.shift(1).rolling(WINDOW, min_periods=1).mean())
team_long["form_ga_avg"] = grouped["ga"].apply(lambda s: s.shift(1).rolling(WINDOW, min_periods=1).mean())
team_long["matches_played_before"] = grouped.cumcount()

team_long[["team", "date", "form_points_avg", "form_gf_avg", "form_ga_avg", "matches_played_before"]].head()


,team,date,form_points_avg,form_gf_avg,form_ga_avg,matches_played_before
0,Abkhazia,2012-09-25,NaN,NaN,NaN,0
1,Abkhazia,2012-10-21,1.000000,1.000000,1.000000,1
2,Abkhazia,2013-09-23,0.500000,0.500000,2.000000,2
3,Abkhazia,2014-06-01,1.333333,1.333333,1.333333,3
4,Abkhazia,2014-06-02,1.250000,1.250000,1.250000,4


In [9]:
# Se reconstruye el formato ancho (una fila por partido) uniendo la forma de local y visitante.
form_cols = ["match_id", "team", "form_points_avg", "form_gf_avg", "form_ga_avg", "matches_played_before"]

home_form = team_long.loc[team_long["is_home"] == 1, form_cols].rename(
    columns={c: f"home_{c}" for c in form_cols if c not in ("match_id", "team")}
).drop(columns="team")

away_form = team_long.loc[team_long["is_home"] == 0, form_cols].rename(
    columns={c: f"away_{c}" for c in form_cols if c not in ("match_id", "team")}
).drop(columns="team")

matches = matches.merge(home_form, on="match_id", how="left").merge(away_form, on="match_id", how="left")

# Partidos donde alguno de los dos equipos no tiene historial previo suficiente se excluyen del
# entrenamiento (no del dataset completo) porque su "forma reciente" no es confiable.
matches["has_min_history"] = (matches["home_matches_played_before"] >= 5) & (matches["away_matches_played_before"] >= 5)
matches["has_min_history"].value_counts()


has_min_history
True     48191
False     1314
Name: count, dtype: int64

### 3.5 Fuerza relativa Elo (histórica, pre-partido) y ranking FIFA (foto actual)

`elo_ratings.csv` resultó ser una serie histórica: una fila por selección cada vez que su rating
cambió, no una foto única. Eso permite unir, para cada partido, el rating Elo **inmediatamente
anterior** a la fecha del partido (`merge_asof` con `direction="backward"` y
`allow_exact_matches=False`, para no incluir un rating actualizado el mismo día del partido). Esto sí
es una variable de entrenamiento válida a nivel de partido, sin fuga de información.

`fifa_rank`, en cambio, es una única foto (2026-07-08). No es correcto unirla a partidos históricos
como si reflejara la fuerza del equipo *en ese momento del pasado*; por eso el ranking FIFA solo se usa
en `team_features_2026.csv`, como parte de la foto actual para simular el Mundial 2026 hacia adelante.

**Limitación adicional detectada:** pese al nombre del archivo (`elo_ratings_2026_07_08.csv`), el
último rating disponible por selección es de diciembre de 2025, no de julio de 2026 — hay un desfase
de aproximadamente 7 meses entre el corte real de Elo y el corte del ranking FIFA. Se documenta como
limitación de `team_features_2026.csv`: la fuerza "actual" de cada selección combina un ranking FIFA
de julio 2026 con un Elo de diciembre 2025.


In [10]:
elo_sorted = elo.sort_values("date")

def elo_before_match(team_col, out_col):
    left = matches[["match_id", "date", team_col]].rename(columns={team_col: "team"}).sort_values("date")
    merged = pd.merge_asof(
        left, elo_sorted, on="date", by="team",
        direction="backward", allow_exact_matches=False,
    )
    return merged.set_index("match_id")["rating"].rename(out_col)

home_elo = elo_before_match("home_team", "home_elo_pre_match")
away_elo = elo_before_match("away_team", "away_elo_pre_match")

matches = matches.join(home_elo, on="match_id").join(away_elo, on="match_id")
matches["elo_diff"] = matches["home_elo_pre_match"] - matches["away_elo_pre_match"]

matches[["date", "home_team", "away_team", "home_elo_pre_match", "away_elo_pre_match", "elo_diff"]].tail()


,date,home_team,away_team,home_elo_pre_match,away_elo_pre_match,elo_diff
49500,2026-07-07,Switzerland,Colombia,1897.0,1998.0,-101.0
49501,2026-07-09,France,Morocco,2062.0,1830.0,232.0
49502,2026-07-10,Spain,Belgium,2171.0,1849.0,322.0
49503,2026-07-11,Norway,England,1922.0,2042.0,-120.0
49504,2026-07-11,Argentina,Switzerland,2113.0,1897.0,216.0


### 3.6 Selección final de variables y exportación

`clean_model_dataset.csv` (nivel partido, para entrenar el modelo de Persona 3):


In [11]:
model_cols = [
    "match_id", "date", "home_team", "away_team",
    "home_score", "away_score", "goal_diff", "result",
    "neutral", "tournament", "tournament_tier", "decided_by_penalties",
    "home_form_points_avg", "home_form_gf_avg", "home_form_ga_avg", "home_matches_played_before",
    "away_form_points_avg", "away_form_gf_avg", "away_form_ga_avg", "away_matches_played_before",
    "home_elo_pre_match", "away_elo_pre_match", "elo_diff",
    "has_min_history",
]

clean_model_dataset = matches[model_cols].copy()
clean_model_dataset.to_csv(PROCESSED_DIR / "clean_model_dataset.csv", index=False)
print("Guardado:", PROCESSED_DIR / "clean_model_dataset.csv", clean_model_dataset.shape)
clean_model_dataset.head()


Guardado: ..\data\processed\clean_model_dataset.csv (49505, 24)


,match_id,date,home_team,away_team,home_score,away_score,goal_diff,result,neutral,tournament,tournament_tier,decided_by_penalties,home_form_points_avg,home_form_gf_avg,home_form_ga_avg,home_matches_played_before,away_form_points_avg,away_form_gf_avg,away_form_ga_avg,away_matches_played_before,home_elo_pre_match,away_elo_pre_match,elo_diff,has_min_history
0,0,1872-11-30,Scotland,England,0.0,0.0,0.0,draw,0,Friendly,1,False,NaN,NaN,NaN,0,NaN,NaN,NaN,0,NaN,NaN,NaN,False
1,1,1873-03-08,England,Scotland,4.0,2.0,2.0,home_win,0,Friendly,1,False,1.000000,0.000000,0.000000,1,1.000000,0.000000,0.000000,1,2003.0,1997.0,6.0,False
2,2,1874-03-07,Scotland,England,2.0,1.0,1.0,home_win,0,Friendly,1,False,0.500000,1.000000,2.000000,2,2.000000,2.000000,1.000000,2,1986.0,2014.0,-28.0,False
3,3,1875-03-06,England,Scotland,2.0,2.0,0.0,draw,0,Friendly,1,False,1.333333,1.666667,1.333333,3,1.333333,1.333333,1.666667,3,2006.0,1994.0,12.0,False
4,4,1876-03-04,Scotland,England,3.0,0.0,3.0,home_win,0,Friendly,1,False,1.250000,1.500000,1.750000,4,1.250000,1.750000,1.500000,4,1997.0,2003.0,-6.0,False


`team_features_2026.csv` (nivel selección, foto actual para simular el Mundial 2026): ranking FIFA
(única foto disponible), el rating Elo más reciente de cada selección hasta la fecha de corte, y su
forma reciente de los últimos `WINDOW` partidos.


In [12]:
fifa_slim = fifa_rank[["team", "rank", "total_points", "confederation"]].rename(
    columns={"rank": "fifa_rank", "total_points": "fifa_points"}
)

latest_elo = (
    elo_sorted.sort_values("date")
    .groupby("team")
    .tail(1)[["team", "date", "rating"]]
    .rename(columns={"date": "elo_date", "rating": "elo_rating"})
)

latest_form = (
    team_long.sort_values("date")
    .groupby("team")
    .tail(WINDOW)
    .groupby("team")
    .agg(recent_points_avg=("points", "mean"),
         recent_gf_avg=("gf", "mean"),
         recent_ga_avg=("ga", "mean"),
         last_match_date=("date", "max"))
    .reset_index()
)

team_features_2026 = (
    fifa_slim
    .merge(latest_elo, on="team", how="left")
    .merge(latest_form, on="team", how="left")
)
team_features_2026.to_csv(PROCESSED_DIR / "team_features_2026.csv", index=False)
print("Guardado:", PROCESSED_DIR / "team_features_2026.csv", team_features_2026.shape)
team_features_2026.sort_values("fifa_rank").head(10)


Guardado: ..\data\processed\team_features_2026.csv (211, 10)


,team,fifa_rank,fifa_points,confederation,elo_date,elo_rating,recent_points_avg,recent_gf_avg,recent_ga_avg,last_match_date
0,France,1,1925.861,UEFA,2025-12-13,2062.0,2.5,2.555556,0.777778,2026-07-09
1,Argentina,2,1925.149,CONMEBOL,2025-12-13,2113.0,2.8,2.888889,0.666667,2026-07-11
2,Spain,3,1912.339,UEFA,2025-12-13,2171.0,2.2,1.777778,0.222222,2026-07-10
3,England,4,1871.385,UEFA,2025-12-13,2042.0,2.1,1.777778,0.777778,2026-07-11
4,Brazil,5,1804.923,CONMEBOL,2025-12-13,1979.0,2.0,2.300000,1.100000,2026-07-05
5,Morocco,6,1803.988,CAF,2025-12-13,1830.0,2.4,2.444444,0.666667,2026-07-09
6,Portugal,7,1787.855,UEFA,2025-12-13,1976.0,2.1,2.300000,0.600000,2026-07-06
7,Belgium,8,1778.355,UEFA,2025-12-13,1849.0,2.2,2.888889,0.888889,2026-07-10
8,Netherlands,9,1775.542,UEFA,2025-12-13,1959.0,1.8,2.100000,1.000000,2026-06-29
9,Mexico,10,1754.297,CONCACAF,2025-12-13,1835.0,2.3,1.900000,0.500000,2026-07-05


## 4. Resumen y entrega a Persona 3

**Dataset limpio entregado:**

- `data/processed/clean_model_dataset.csv` — un registro por partido histórico, con resultado,
  diferencia de goles, contexto (sede neutral, importancia de torneo), forma reciente relativa de
  ambos equipos y **diferencia de rating Elo pre-partido**, todo calculado sin fuga de información.
- `data/processed/team_features_2026.csv` — una fila por selección con su ranking FIFA (foto actual),
  rating Elo más reciente y forma reciente, para inicializar la simulación del Mundial 2026.

**Variables descartadas y por qué:**

- `goalscorers.csv` y los datasets de Transfermarkt (`opcional/`): no aportan al modelo base y añaden
  complejidad de limpieza (nombres de jugadores, valores de mercado) desproporcionada al tiempo
  disponible. Quedan documentados como fuente opcional.
- Ranking FIFA por fecha de partido: no disponible (solo hay una foto del 2026-07-08); por eso no se
  usa como variable de entrenamiento a nivel de partido, solo como variable de estado actual para la
  simulación hacia adelante. El rating Elo sí pudo incorporarse de forma histórica porque, a diferencia
  de lo previsto inicialmente, el archivo descargado es una serie temporal completa y no una foto.

**Pendiente / recomendación para Persona 3:** decidir si el modelo de resultado de partido se entrena
solo con `clean_model_dataset.csv` (histórico, incluye `elo_diff`) y luego se alimenta con
`team_features_2026.csv` para estimar la fuerza relativa de los cruces del Mundial 2026, o si se
combinan ambas fuentes en un único esquema de features. Ambos archivos comparten la columna `team`
con nombres ya normalizados con `TEAM_NAME_MAP`.
